# [stand-in] CubeLang emitter — SFT with Unsloth, export to GGUF

**What this is.** A small open model fine-tuned to emit CubeLang programs from a question, so the
serve stack (VM verification, worlds, harvest, the learning gate as a live loop) can be built while
the 2B CubbyLLM trunk trains. **It is a stand-in**: nothing it measures is a CubbyLLM result — no
hybrid backbone, no θ=f(c), no bounded state, no episodic store. Tag every number it produces
`[stand-in]`; it lives behind the trunk interface in `standin/`, outside `cubbyllm/`, and is swapped
out the day the 2B checkpoint exists.

**Data.** `standin/data/out/emitter_sft.jsonl`, built by `standin/data/build_emitter_sft.py` and
re-verified program-by-program through the real Rust cubelang VM: GSM8K-*train*-derived arithmetic
programs (GSM8K test excluded — it is H-G4's eval), role-binding programs (capped), the four
reasoning kernels, and our 517 verified multi-hop chain programs (`ISolve`/`recover` dialect — the
ones the oracle actually needs). Upload the jsonl + manifest to `Drive/cubbyllm/standin/` first.

**Model.** `LiquidAI/LFM2.5-2.6B` (2026-08 Hub check: best structured-output profile among ≤4B —
IFStruct 85.5 / Multi-IF 80.1 / BFCL v4 56.9 self-reported; LFM Open License, fine for a non-commercial
prototype). Fallback: `ibm-granite/granite-4.2-3b` (Apache-2.0, stronger raw code) once Unsloth artifacts land.

**Identity + hormones.** The set also carries ~300 identity turns (Cubby / Grillcheese Research Lab / not AGI /
how it answers) under their OWN system prompt, which includes a sampled **hormonal state** (dopamine,
serotonin, cortisol, oxytocin, noradrenaline — the cubbyverse demo's `neurochemistry.py` set and bands)
and the register it implies. The host injects that block at serve time, so "everything is modulated by
hormones" is true by construction; tone/caution change, facts never do; the emitter turns carry no state.
Identity turns are **bilingual (EN/FR)**: a French question gets a French answer, and the don't-know line
is verbatim in the question's language. Every record names its `system` prompt — the formatter below uses it.

**v2 (after the first run, 2026-08-30).** LFM2.5 opens a `<think>` block on its own and confabulated context in it;
the targets now start with an empty `<think>\n</think>\n` and inference prefills the same, so the answer starts
immediately. Role-binding is cut to the curated `svc` set (chat fillers dropped, cap 1,500) and every such prompt is
wrapped as an explicit task ("Record this as an event: …") so a bare sentence never means "bind it"; chains are
upsampled ×3 in train.

**Eval split.** Colab does text exact-match on the held-out `val` split (cheap, format-level).
The real read — *does the emitted program execute on the VM and hit gold* — runs **locally** after
this notebook exports the GGUF (`standin/eval_emitter_vm.py`, llama.cpp Vulkan + `cubelang.exe`).

**v4 (2026-09-02).** `emitter_sft_v4.jsonl` = the whole v3 set as replay (so nothing learned is lost) + ~2.2k programs the GAME generated and the VM verified (`standin/data/build_game_sft.py`): flee/go decisions and safer-exit/shorter-path comparisons on live numbers, 1–2-hop neighbor chains over level-scoped cells, exit counts, superpower compositions — the families the forge probe measured v3 at 0.75 / 0.50 / 0.00 on. Set `VERSION` in the setup cell (`v4` default). After export: locally `eval_emitter_vm.py --data standin/data/out/emitter_sft_v4.jsonl --gguf <v4 gguf>` and `scripts/forge_probe.py --gguf <v4 gguf>` — the before/after acceptance is the result.


In [ ]:
# --- setup (run once per session) ---
import os, json, time, random, re
!pip -q install unsloth trl datasets
import sys
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm/standin'
# the repo checkout (loud: a failed clone/pull used to hide behind -q and surface as 'cannot find identity')
if not os.path.exists('/content/CubbyLLM/.git'):
    !rm -rf /content/CubbyLLM && git clone https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM 2>&1 | tail -2
else:
    !cd /content/CubbyLLM && git fetch origin 2>&1 | tail -1 && git reset -q --hard origin/master && git log -1 --format='repo at %h %s'
sys.path.insert(0, '/content/CubbyLLM/standin/data'); sys.path.insert(0, '/content/CubbyLLM')
try:
    from identity import identity_ok, load_facts, EMITTER_SYSTEM, voice_ok, is_model_guard, is_identity_reply   # the identity check + facts
    print('identity: from the repo checkout')
except ModuleNotFoundError:                     # no checkout (private repo / network): Drive carries a copy
    assert os.path.exists(f'{DRIVE}/identity.py'), f'identity.py not in the repo checkout nor at {DRIVE}'
    sys.path.insert(0, DRIVE)
    from identity import identity_ok, load_facts, EMITTER_SYSTEM, voice_ok, is_model_guard, is_identity_reply
    print('identity: from Drive (repo checkout missing)')
FACTS = load_facts()
VERSION = os.environ.get('STANDIN_VERSION', 'v4')   # v3 = the regen/harvest mix; v4 = v3 replay + the GAME's VM-verified programs
SUFFIX = '' if VERSION == 'v3' else f'_{VERSION}'
DATA = f'{DRIVE}/emitter_sft{SUFFIX}.jsonl'
MANIFEST = f'{DRIVE}/emitter_sft{SUFFIX}.manifest.json'
OUT = f'{DRIVE}/emitter_lfm25_2p6b{SUFFIX}'          # adapter + merged + GGUF land here
MODEL = 'LiquidAI/LFM2.5-2.6B'
MAX_SEQ = 2048
!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in (DATA, MANIFEST):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)
m = json.load(open(MANIFEST))   # v3 manifests say n_records_kept, v4/v5 say n_records
print('manifest', m.get('version', 'v3'), ':', m['by_task'], '| records', m.get('n_records', m.get('n_records_kept')),
      '| built', m['built'][:19], '| git', m['git_rev'][:8])

In [ ]:
# --- data: chat-format the records; train/val from the builder's deterministic split ---
SYSTEM = EMITTER_SYSTEM   # emitter turns; identity turns carry their own r['system'] with the hormonal state
recs = [json.loads(l) for l in open(DATA, encoding='utf-8')]
recs = [r for r in recs if r.get('vm_ok') in (True, None) and r.get('gold_match') is not False]
train = [r for r in recs if r['split'] == 'train']; val = [r for r in recs if r['split'] == 'val']
from collections import Counter
print('train', len(train), Counter(r['task'] for r in train)); print('val  ', len(val), Counter(r['task'] for r in val))

NO_THINK = '<think>\n</think>\n'   # LFM2.5 opens <think> on its own; train it to close immediately (v2, 2026-08-30)
def to_messages(r):
    return [{'role': 'system', 'content': r.get('system') or SYSTEM},
            {'role': 'user', 'content': r['prompt']},
            {'role': 'assistant', 'content': NO_THINK + r['program'].strip() + '\n'}]

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAX_SEQ, load_in_4bit=False, dtype=None)

def fmt(r):
    return {'text': tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False)}
from datasets import Dataset
random.Random(0).shuffle(train)
ds_train = Dataset.from_list([fmt(r) for r in train for _ in range(int(r.get('repeat', 1)))])   # chains x3 (builder's `repeat`)
print('train rows after repeat weights:', len(ds_train))
lens = [len(tokenizer(x['text']).input_ids) for x in ds_train.select(range(min(500, len(ds_train))))]
print('token lengths (sample of 500): max', max(lens), 'p95', sorted(lens)[int(0.95*len(lens))], '-> MAX_SEQ', MAX_SEQ)
print(ds_train[0]['text'][:900])

In [ ]:
# --- LoRA + SFT (one epoch; ~4-5M tokens; minutes on an A100) ---
from trl import SFTTrainer, SFTConfig
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj', 'o_proj', 'in_proj', 'w1', 'w2', 'w3', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=0)
cfg = SFTConfig(output_dir='/content/emitter_ckpt', per_device_train_batch_size=8, gradient_accumulation_steps=4,
                num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20,
                logging_steps=10, save_strategy='no', bf16=True, max_seq_length=MAX_SEQ, dataset_text_field='text',
                packing=False, report_to='none', seed=0)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, args=cfg)
t0 = time.time(); stats = trainer.train(); print(f'trained in {(time.time()-t0)/60:.1f} min; final loss', stats.training_loss)
os.makedirs(OUT, exist_ok=True); model.save_pretrained(f'{OUT}/adapter'); tokenizer.save_pretrained(f'{OUT}/adapter')
print('adapter ->', f'{OUT}/adapter')

In [ ]:
# --- export FIRST: merged fp16 + GGUF (q8_0 for the VM eval, q4_k_m for the 12 GB Vulkan box) ---
model.save_pretrained_merged(f'{OUT}/merged', tokenizer, save_method='merged_16bit')
model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method=['q8_0', 'q4_k_m'])
!ls -la {OUT}/gguf
print('done -> download the GGUF, then locally: python standin/eval_emitter_vm.py --gguf <file> --val-generations', f'{OUT}/val_generations.json')

In [ ]:
# --- OPTIONAL quick format-level eval (a smoke test; the GGUF above is already exported) ---
# NOT the verified read (the Rust VM is not on Colab): run standin/eval_emitter_vm.py locally on the GGUF for that.
# Kept small on purpose: a stratified handful per task with per-task token caps (v4's 200 x 1200-token eval took
# longer than the training). Set N_PER_TASK = 0 to skip.
FastLanguageModel.for_inference(model)
import transformers; transformers.logging.set_verbosity_error()
def strip_think(s):   # LFM2.5's template opens a <think> block; everything up to </think> is reasoning, not the answer
    return re.sub(r'^\s*(?:<think>)?.*?</think>\s*', '', s, count=1, flags=re.S) if '</think>' in s else s
def norm(s): return re.sub(r'\s+', ' ', re.sub(r'#.*', '', strip_think(s))).strip()   # drop think + comments, collapse whitespace
def emit(prompt, max_new=768, system=None):
    text = tokenizer.apply_chat_template([{'role': 'system', 'content': system or SYSTEM}, {'role': 'user', 'content': prompt}],
                                         tokenize=False, add_generation_prompt=True) + NO_THINK   # prefill: no reasoning
    enc = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)   # raw (think block kept in the file)
N_PER_TASK = 6
rng = random.Random(1); by_task = {}
for r in val: by_task.setdefault(r['task'], []).append(r)
sample = [r for t, rs in sorted(by_task.items()) for r in rng.sample(rs, min(N_PER_TASK, len(rs)))]
CAP = {'chain': 500, 'kernel': 600, 'arithmetic': 500, 'role_binding': 400, 'identity': 200, 'chat': 120, 'content': 40, 'emotion': 40}
print('eval sample', len(sample), {t: min(N_PER_TASK, len(rs)) for t, rs in sorted(by_task.items())})
hits = Counter(); tot = Counter(); outputs = []
t0 = time.time()
for i, r in enumerate(sample):
    if r['task'] == 'identity':
        gen = emit(r['prompt'], max_new=CAP['identity'], system=r.get('system')); ok = identity_ok(r.get('subtype', ''), strip_think(gen).strip(), FACTS, r.get('lang', 'en'))
    elif r['task'] in ('chat', 'content', 'emotion'):   # v5: conversational tasks -- their own system prompt, their own check
        gen = emit(r['prompt'], max_new=CAP.get(r['task'], 120), system=r.get('system')); g = strip_think(gen).strip()
        if r['task'] == 'chat':      ok = bool(g) and voice_ok(g, FACTS) and not is_model_guard(g) and not is_identity_reply(g, FACTS)
        elif r['task'] == 'emotion': ok = g.lower().replace('—', ',').split(',')[0].strip(' -:.') in {str(x).lower() for x in (r.get('gold_any') or [r.get('gold')])}
        else:                        ok = g.lower().split(' ')[0].strip(' —-:.,') == str(r.get('gold')).lower()
    else:
        gen = emit(r['prompt'], max_new=CAP.get(r['task'], 600)); ok = norm(gen) == norm(r['program'])   # empty think is trained: no room needed
    tot[r['task']] += 1; hits[r['task']] += int(ok)
    outputs.append({'id': r['id'], 'task': r['task'], 'subtype': r.get('subtype', ''), 'prompt': r['prompt'],
                    'reference': r['program'], 'generated': gen, 'exact_match': ok, 'gold': r.get('gold'),
                    'system': r.get('system'), 'lang': r.get('lang')})
    if (i+1) % 50 == 0: print(f'  {i+1}/{len(sample)} ({time.time()-t0:.0f}s)')
print('[stand-in] val by task (identity = identity_ok; chat = voice/guard/no-bio; content/emotion = label first; programs = text exact-match):', {t: f'{hits[t]}/{tot[t]}' for t in tot}, '| overall', sum(hits.values())/sum(tot.values()))
json.dump({'model': MODEL, 'n': len(sample), 'exact_match_by_task': {t: hits[t]/tot[t] for t in tot},
           'outputs': outputs, 'manifest_output_sha256': m['output_sha256']}, open(f'{OUT}/val_generations.json', 'w'), indent=1)
print('generations ->', f'{OUT}/val_generations.json  (the local VM eval reads this file too)')

### How to read

- **Exact match is a format read, not a capability read.** Arithmetic programs have one canonical
  decomposition per question in the data, so EM is meaningful there; role-binding EM mostly tests
  whether the model picked the same ACTION/AGENT split; chain programs are deterministic given the
  retrieved facts, so EM is fair. Anything the model emits that *differs but executes to gold* only shows
  up in the local VM eval — expect VM-verified accuracy ≥ EM.
- **Report per task.** The chain task is the one the oracle needs; a high overall EM carried by
  role-binding is not a result.
- **Identity is scored by `identity_ok`**, not exact match: name/builder present, never affirms AGI or
  feelings, never claims to be another model, and the affect turns must speak in state/register language
  that matches the injected hormonal block. Try a few live prompts with different `state` blocks to see the
  register shift (cautious vs curious vs warm).
- **LFM2.5 thinks before it answers** (`<think>…</think>` from the chat template). The raw generation is
  saved; scoring strips the block (`strip_think`) — locally `eval_emitter_vm.py` does the same. If the think
  block routinely exceeds a few hundred tokens the emitter is wasting budget: consider SFT with the think
  block trained empty, or a template that disables it.
- **Everything here is `[stand-in]`.** It goes in `standin/README.md` and the TODO, never in
  `CUBBYLLM_HYPOTHESES.md` except as a pointer.